<h1 style = 'color: red'>Interactive Visual Analytics with Folium</h1>

<h2 style = 'color: yellow'>Task1: mark all launch sites on a map</h2>
<h2 style = 'color: yellow'>Task2: mark the success / failed launches for each site on the map</h2>
<h2 style = 'color: yellow'>Task3: calculate the distances between a launch site to its proximities</h2>

In [1]:
import folium
import pandas as pd
# import folium markercluster plugin
from folium.plugins import MarkerCluster
# import folium mouseposition plugin
from folium.plugins import MousePosition
# import folium divicon plugin
from folium.features import DivIcon

In [2]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
df = pd.read_csv(URL)
df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Flight Number      56 non-null     int64  
 1   Date               56 non-null     object 
 2   Time (UTC)         56 non-null     object 
 3   Booster Version    56 non-null     object 
 4   Launch Site        56 non-null     object 
 5   Payload            56 non-null     object 
 6   Payload Mass (kg)  56 non-null     float64
 7   Orbit              56 non-null     object 
 8   Customer           56 non-null     object 
 9   Landing Outcome    56 non-null     object 
 10  class              56 non-null     int64  
 11  Lat                56 non-null     float64
 12  Long               56 non-null     float64
dtypes: float64(3), int64(2), object(8)
memory usage: 5.8+ KB


In [10]:
# select relevant sub-columns
df = df[['Launch Site', 'Lat', 'Long', 'class']]
total_launch = [0, 0, 0, 0]
success_launch = [0, 0, 0, 0]
failed_launch = [0, 0, 0, 0]
df_2 = df.groupby('Launch Site', as_index = False).first()
launch_site = df_2['Launch Site'].unique()
i = 0
for site in launch_site:
    i += 1
    for _, row in df.iterrows():
        if row['Launch Site'] == site:
            total_launch[i - 1] += 1
            if row['class'] == 1:
                success_launch[i - 1] += 1
    failed_launch[i - 1] = total_launch[i - 1] - success_launch[i - 1]
print(launch_site)
print(total_launch)
print(success_launch)
df_2['Success'] = success_launch
df_2['Failed'] = failed_launch
df_2

['CCAFS LC-40' 'CCAFS SLC-40' 'KSC LC-39A' 'VAFB SLC-4E']
[26, 7, 13, 10]
[7, 3, 10, 4]


,Launch Site,Lat,Long,class,Success,Failed
0,CCAFS LC-40,28.562302,-80.577356,0,7,19
1,CCAFS SLC-40,28.563197,-80.576820,1,3,4
2,KSC LC-39A,28.573255,-80.646895,1,10,3
3,VAFB SLC-4E,34.632834,-120.610745,0,4,6


In [11]:
# start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location = nasa_coordinate, zoom_start = 10)

In [12]:
site_map

In [13]:
# create a blue circle at NASA Center's corrdinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius = 100, color = '#d35400', fill = True).add_child(folium.Popup('NASA Johnson Space Center'))
# create a blue circle at NASA Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # create an icon as a text label
    icon = DivIcon(
        icon_size = (20, 20),
        icon_anchor = (0, 0),
        html = '<div style = "font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC'
    )
)
site_map.add_child(circle)
site_map.add_child(marker)
site_map

In [16]:
# loop through dataframe to input coordinates onto map using folium
for _, row in df_2.iterrows():
    circle = folium.Circle([row['Lat'], row['Long']], radius = 100, color = '#000000', fill = False, fill_opacity = 0).add_child(folium.Popup(f'Success: {row['Success']}\n Failed: {row['Failed']}'))
    marker = folium.map.Marker(
        location = [row['Lat'], row['Long']],
        icon = DivIcon(
            icon_size = (20, 20),
            icon_anchor = (0, 0),
            html = '<div style="font-size: 12; color:#276FF5;"><b>%s</b></div>' % str(row["Launch Site"]),
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)
site_map

In [21]:
marker_cluster = MarkerCluster().add_to(site_map)
color = []
for _, row in df.iterrows():
    if row['class'] == 0:
        color.append('red')
    else:
        color.append('green')
df['marker_color'] = color
for _, row in df.iterrows():
    marker = folium.Marker(
        location = [row['Lat'], row['Long']],
    ).add_to(marker_cluster)
site_map

In [22]:
# add mouse position to get the coordinate (lat, long) for a mouse over at the map
formatter = 'function(num) {return L.Util.formatNum(num, 5);};'
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [23]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance